# SDN-FL IDS trên CICIoV — class-incremental

Hbaieb, Ayed, Chaari, *A federated learning based IDS approach for the IoV*,
ARES 2022.

Bốn lượt chạy: **CNN+trust** (chính), **RNN+trust**, **CNN+FedAvg thường**
(đối chứng để trả lời câu hỏi cơ chế trust có tác dụng gì không), và
**Random Forest**.

**Cần trước khi chạy:** Kaggle Dataset `iov-100client` chứa thư mục `100client`
(giữ thư mục con `federated_data/`). Bật **GPU T4**.

## 1. Setup

In [ ]:
import os, glob, time

REPO = "SDNFL-IDS"
CODE = f"/kaggle/working/{REPO}"
DATA = "/kaggle/input/iov-100client"     # sửa cho khớp tên Dataset của bạn

if not os.path.isdir(CODE):
    !git clone -q https://github.com/TongXuanVu/{REPO}.git {CODE}
else:
    !cd {CODE} && git pull -q

!pip install -q "flwr[simulation]"
import flwr, torch
print("flwr", flwr.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.is_available())

## 2. Kiểm tra dữ liệu — đừng bỏ qua

In [ ]:
fed = os.path.join(DATA, "federated_data")
assert os.path.isdir(fed), f"Khong thay {fed}. Sua bien DATA, hoac Dataset chua gan vao notebook."

shards = sorted(glob.glob(os.path.join(fed, "*.pt")))
print(f"{len(shards)} shard (ky vong 500 cho bo 100client)")

blob = torch.load(os.path.join(DATA, "global_test_data.pt"),
                  map_location="cpu", weights_only=False)
x, y = blob["x"], blob["y"]
print(f"global test : x={tuple(x.shape)} {x.dtype} | so lop={int(y.max()) + 1}")

assert x.shape[1] == 31, f"So dac trung = {x.shape[1]}, khong phai 31 -> phai sua INPUT_LEN"
assert int(y.max()) + 1 <= 13, "Nhieu hon 13 lop -> phai sua NUM_GLOBAL_CLASSES"
print("\nDu lieu OK.")

## 3. Chạy thử nhanh (~3 phút)

Bắt buộc chạy trước khi chạy thật.

In [ ]:
!cd {CODE} && python run_sim.py --data-dir {DATA} \
    --out-dir /kaggle/working/_thu --clients 100 --rounds 2 --tasks 0,1 \
    --max-samples 20000 --test-samples 20000 \
    --arch cnn --weighting trust --simulate-sdn 2>&1 | tail -20

## 4. Chạy thật — **cứ chạy lại cell này mỗi khi Kaggle hết giờ**

Mỗi round đều ghi thẳng xuống đĩa ngay: `metrics_task*.csv`, `checkpoints/round_NNN.pth`,
`_logs/*.log`. Session bị cắt giữa chừng không mất gì.

Session sau, **chạy lại đúng cell này**: nó đếm số dòng trong CSV, bỏ qua task đã đủ
round, và chỉ chạy tiếp số round còn thiếu của task đang dở. Round vẫn đánh liên tục,
CSV không có dòng trùng.

`--cm-every 5` ghi confusion matrix mỗi 5 round, để bị cắt giữa task vẫn có bản gần nhất.

> Muốn xoá sạch làm lại: đổi `--out-dir`, hoặc xoá thư mục đó rồi thêm `--restart`.


In [ ]:
COMMON = (f"--data-dir {DATA} --clients 100 --rounds 30 "
          "--batch-size 512 --max-samples 0 --cm-every 5")
print(COMMON)


In [ ]:
# (a) 1D-CNN + trọng số trust — cấu hình chính của bài
t0 = time.time()
!cd {CODE} && python run_sim.py {COMMON} \
    --out-dir /kaggle/working/out_cnn_trust \
    --arch cnn --weighting trust --simulate-sdn
print(f"\nCNN+trust xong sau {(time.time() - t0) / 60:.1f} phut")

In [ ]:
# (b) 1D-RNN + trọng số trust
t0 = time.time()
!cd {CODE} && python run_sim.py {COMMON} \
    --out-dir /kaggle/working/out_rnn_trust \
    --arch rnn --weighting trust --simulate-sdn
print(f"\nRNN+trust xong sau {(time.time() - t0) / 60:.1f} phut")

In [ ]:
# (c) ĐỐI CHỨNG: FedAvg thường, trọng số chỉ theo số mẫu.
#     So (a) với (c) mới trả lời được: cơ chế trust của bài có cải thiện gì không?
t0 = time.time()
!cd {CODE} && python run_sim.py {COMMON} \
    --out-dir /kaggle/working/out_cnn_fedavg \
    --arch cnn --weighting samples --simulate-sdn
print(f"\nCNN+FedAvg xong sau {(time.time() - t0) / 60:.1f} phut")

In [ ]:
# (d) Random Forest — không FedAvg được nên ghép cây theo trọng số.
#     Chạy trên toàn bộ dữ liệu (không class-incremental) làm mốc tham chiếu.
!cd {CODE} && python rf_baseline.py \
    --data-dir {DATA} --out-dir /kaggle/working/out_rf \
    --clients 0 1 2 3 4 5 6 7 8 9 --simulate-sdn \
    --trees-local 100 --trees-total 300 2>&1 | tail -20

## 5. Gộp kết quả + đo mức độ quên

In [ ]:
!cd {CODE} && python collect_results.py --out-dir /kaggle/working/ket_qua \
    --runs CNN-trust=/kaggle/working/out_cnn_trust \
           RNN-trust=/kaggle/working/out_rnn_trust \
           CNN-fedavg=/kaggle/working/out_cnn_fedavg

In [ ]:
import pandas as pd
from IPython.display import display, Image

K = "/kaggle/working/ket_qua"
print("=== So sanh 3 cau hinh FL ==="); display(pd.read_csv(f"{K}/comparison.csv"))
print("=== Muc do quen ===");           display(pd.read_csv(f"{K}/forgetting.csv"))
display(Image(f"{K}/accuracy_curve.png"))

In [ ]:
# Trọng số từng controller — dùng vẽ hình chứng minh cơ chế trust
w = pd.read_csv("/kaggle/working/out_cnn_trust/client_weights_cnn_task4.csv")
agg = w.groupby("client")[["throughput_mbps", "latency_ms", "node_trust",
                           "quality", "behaviour", "weight"]].mean()
display(agg.sort_values("weight", ascending=False))

ax = agg.sort_values("weight").plot.barh(y="weight", figsize=(7, 4), legend=False)
ax.set_xlabel("Trong so tong hop trung binh")
ax.set_ylabel("Controller")
ax.set_title("Trust weighting: controller nao duoc tin hon")

## 6. Đóng gói tải về

In [ ]:
# Kiem tra truoc khi tai ve: du dong metric? du checkpoint? du log?
import glob, os

for f in sorted(glob.glob("/kaggle/working/ket_qua/metrics_all_*.csv")):
    n = sum(1 for _ in open(f)) - 1
    print(f"{os.path.basename(f):32s} {n:4d} dong metric  (ky vong 150 = 30 round x 5 task)")

for d in sorted(glob.glob("/kaggle/working/out_*/checkpoints*")):
    print(f"{d:52s} {len(glob.glob(d + '/round_*.pth')):4d} checkpoint")

print(f"\n{len(glob.glob('/kaggle/working/out_*/_logs/*.log'))} file log")
print(f"{len(glob.glob('/kaggle/working/out_*/confusion_matrix_*.csv'))} confusion matrix CSV")


In [ ]:
# Dong goi DAY DU: checkpoint (.pth) + log + CSV + confusion matrix
!cd /kaggle/working && zip -qr ket_qua_p4.zip \
    ket_qua out_cnn_trust out_rnn_trust out_cnn_fedavg out_rf
!du -sh /kaggle/working/out_cnn_trust /kaggle/working/out_rnn_trust \
        /kaggle/working/out_cnn_fedavg /kaggle/working/out_rf
!ls -lh /kaggle/working/ket_qua_p4.zip
print("\nTai ve tu tab Output ben phai TRUOC KHI het session.")
